# Machine Learning Assignment: Breast Cancer Diagnosis Classification

### Dataset Overview
In this project, we analyze the **Breast Cancer Wisconsin (Diagnostic)** dataset from Kaggle. The dataset contains diagnostic measurements extracted from digitized images of fine needle aspirates (FNA) of breast masses.

### Task & Workflow
The objective is to train and evaluate supervised classification models to predict whether a breast mass is **Malignant** (cancerous) or **Benign** (non-cancerous).

**Pipeline Stages:**
1. **Data Loading & Inspection**: Load `breast_cancer.csv` and inspect schema.
2. **Data Cleaning & Preprocessing**: Drop identifier columns, verify missing values, and encode target labels.
3. **Train-Test Split & Scaling**: 80/20 stratified split and feature standardization via `StandardScaler`.
4. **Model Training**:
   - **Logistic Regression**: Linear baseline with L2 penalty.
   - **Random Forest Classifier**: Ensemble of 100 decision trees.
5. **Model Evaluation**: Compare performance using Accuracy, Precision, Recall, F1-Score, and Confusion Matrices.
6. **Key Insights**: Discussion of evaluation trade-offs and medical diagnostic considerations.


## 1. Import Required Libraries
We import data handling tools (`pandas`, `numpy`), preprocessing and model modules from `scikit-learn`, and evaluation metrics.


In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

print("Libraries imported successfully.")


Libraries imported successfully.


## 2. Load the Dataset
We load `breast_cancer.csv` into a pandas DataFrame and inspect its dimensions and initial records.


In [2]:
# Load the dataset from local CSV file
df = pd.read_csv('breast_cancer.csv')

print(f"Dataset shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()


Dataset shape: 569 rows, 32 columns


,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,fractal_dimension_mean,radius_se,texture_se,perimeter_se,area_se,smoothness_se,compactness_se,concavity_se,concave points_se,symmetry_se,fractal_dimension_se,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,842000,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,1.0950,0.9053,8.589,153.40,0.006399,0.04904,0.05373,0.01587,0.03003,0.006193,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,842001,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,0.5435,0.7339,3.398,74.08,0.005225,0.01308,0.01860,0.01340,0.01389,0.003532,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,842002,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,0.7456,0.7869,4.585,94.03,0.006150,0.04006,0.03832,0.02058,0.02250,0.004571,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,842003,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,0.4956,1.1560,3.445,27.23,0.009110,0.07458,0.05661,0.01867,0.05963,0.009208,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,842004,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,0.7572,0.7813,5.438,94.44,0.011490,0.02461,0.05688,0.01885,0.01756,0.005115,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


## 3. Data Cleaning and Preprocessing
A critical step in machine learning is data sanitation:
1. **Inspect Missing Values**: Verify if any feature contains null/NaN values.
2. **Remove Non-Predictive Columns**: Drop the `id` column, as patient identifiers carry no clinical relevance and could cause data leakage.
3. **Encode Target Variable**: Map the categorical `diagnosis` column (`M` = Malignant, `B` = Benign) into binary values (`1` for Malignant, `0` for Benign).


In [3]:
# 1. Check for missing values
missing_count = df.isnull().sum().sum()
print(f"Total missing values in dataset: {missing_count}")

# 2. Drop the patient identifier column 'id'
df_clean = df.drop(columns=['id'])
print(f"Shape after dropping 'id': {df_clean.shape}")

# 3. Check target class distribution
print("\nClass distribution before encoding:")
print(df_clean['diagnosis'].value_counts())

# Map 'M' -> 1 (Malignant) and 'B' -> 0 (Benign)
df_clean['diagnosis'] = df_clean['diagnosis'].map({'M': 1, 'B': 0})
print("\nTarget encoded successfully (1 = Malignant, 0 = Benign).")
print(f"Malignant cases: {(df_clean['diagnosis'] == 1).sum()} ({(df_clean['diagnosis'] == 1).mean():.1%})")
print(f"Benign cases:    {(df_clean['diagnosis'] == 0).sum()} ({(df_clean['diagnosis'] == 0).mean():.1%})")


Total missing values in dataset: 0
Shape after dropping 'id': (569, 31)

Class distribution before encoding:
diagnosis
B    357
M    212
Name: count, dtype: int64

Target encoded successfully (1 = Malignant, 0 = Benign).
Malignant cases: 212 (37.3%)
Benign cases:    357 (62.7%)


### Feature Summary Statistics
We inspect the statistical characteristics (mean, std, min, max, quartiles) for the primary mean diagnostic features.


In [4]:
# Inspect statistical summary of selected primary mean features
df_clean[['radius_mean', 'texture_mean', 'perimeter_mean', 'area_mean', 'smoothness_mean']].describe()


,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean
count,569.000000,569.000000,569.000000,569.000000,569.000000
mean,14.127292,19.289649,91.969033,654.889104,0.096360
std,3.524049,4.301036,24.298981,351.914129,0.014064
min,6.981000,9.710000,43.790000,143.500000,0.052630
25%,11.700000,16.170000,75.170000,420.300000,0.086370
50%,13.370000,18.840000,86.240000,551.100000,0.095870
75%,15.780000,21.800000,104.100000,782.700000,0.105300
max,28.110000,39.280000,188.500000,2501.000000,0.163400


## 4. Train-Test Split & Feature Scaling
To evaluate the models on unseen data:
- We separate the independent features ($X$) and target label ($y$).
- We use an **80/20 stratified split** (`stratify=y`) to maintain the ratio of malignant to benign cases in both splits.
- We standardize features using `StandardScaler` ($\mu = 0, \sigma = 1$). Scaling is essential for Logistic Regression to ensure balanced weight updates during optimization.


In [5]:
# Separate features (X) and target (y)
X = df_clean.drop(columns=['diagnosis'])
y = df_clean['diagnosis']

# Perform 80% train / 20% test stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples, {X_train.shape[1]} features")
print(f"Testing set:  {X_test.shape[0]} samples, {X_test.shape[1]} features")

# Scale features based on training statistics
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Standard scaling applied successfully.")


Training set: 455 samples, 30 features
Testing set:  114 samples, 30 features
Standard scaling applied successfully.


## 5. Model Training
We train two distinct machine learning models:
1. **Logistic Regression**: A linear probabilistic classifier optimized using L-BFGS.
2. **Random Forest Classifier**: A non-linear ensemble of 100 decision trees using bootstrap aggregation (bagging).


In [6]:
# 1. Train Logistic Regression model
log_reg = LogisticRegression(random_state=42, max_iter=1000)
log_reg.fit(X_train_scaled, y_train)
y_pred_lr = log_reg.predict(X_test_scaled)

# 2. Train Random Forest Classifier
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42)
rf_clf.fit(X_train, y_train)
y_pred_rf = rf_clf.predict(X_test)

print("Both models trained and test predictions generated successfully.")


Both models trained and test predictions generated successfully.


## 6. Model Evaluation and Comparison
We evaluate both models on test set predictions using:
- **Accuracy**: Overall fraction of correct predictions.
- **Precision**: $\frac{TP}{TP + FP}$ — proportion of predicted malignant tumors that were truly malignant.
- **Recall (Sensitivity)**: $\frac{TP}{TP + FN}$ — proportion of actual malignant tumors detected.
- **F1-Score**: Harmonic mean of Precision and Recall.
- **Confusion Matrix**: Tabulation of True Negatives (TN), False Positives (FP), False Negatives (FN), and True Positives (TP).


In [7]:
# Function to compute evaluation metrics
def get_metrics_dict(model_name, y_true, y_pred):
    return {
        'Model': model_name,
        'Accuracy': f"{accuracy_score(y_true, y_pred) * 100:.2f}%",
        'Precision': f"{precision_score(y_true, y_pred) * 100:.2f}%",
        'Recall': f"{recall_score(y_true, y_pred) * 100:.2f}%",
        'F1-Score': f"{f1_score(y_true, y_pred) * 100:.2f}%"
    }

# Create comparison summary table
comparison_records = [
    get_metrics_dict('Logistic Regression', y_test, y_pred_lr),
    get_metrics_dict('Random Forest Classifier', y_test, y_pred_rf)
]

comparison_df = pd.DataFrame(comparison_records)
print("=== Model Performance Summary ===")
comparison_df


=== Model Performance Summary ===


,Model,Accuracy,Precision,Recall,F1-Score
0,Logistic Regression,96.49%,97.50%,92.86%,95.12%
1,Random Forest Classifier,97.37%,100.00%,92.86%,96.30%


In [8]:
# Detailed Classification Reports and Confusion Matrices
labels = ['Benign (0)', 'Malignant (1)']

print("=" * 65)
print("1. LOGISTIC REGRESSION EVALUATION")
print("=" * 65)
cm_lr = confusion_matrix(y_test, y_pred_lr)
print("Confusion Matrix:")
print(f"  True Negatives (Benign correctly identified):    {cm_lr[0, 0]}")
print(f"  False Positives (Benign mistaken for Malignant): {cm_lr[0, 1]}")
print(f"  False Negatives (Malignant missed):              {cm_lr[1, 0]}")
print(f"  True Positives (Malignant correctly identified): {cm_lr[1, 1]}")
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_lr, target_names=labels))

print("=" * 65)
print("2. RANDOM FOREST CLASSIFIER EVALUATION")
print("=" * 65)
cm_rf = confusion_matrix(y_test, y_pred_rf)
print("Confusion Matrix:")
print(f"  True Negatives (Benign correctly identified):    {cm_rf[0, 0]}")
print(f"  False Positives (Benign mistaken for Malignant): {cm_rf[0, 1]}")
print(f"  False Negatives (Malignant missed):              {cm_rf[1, 0]}")
print(f"  True Positives (Malignant correctly identified): {cm_rf[1, 1]}")
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_rf, target_names=labels))


1. LOGISTIC REGRESSION EVALUATION
Confusion Matrix:
  True Negatives (Benign correctly identified):    71
  False Positives (Benign mistaken for Malignant): 1
  False Negatives (Malignant missed):              3
  True Positives (Malignant correctly identified): 39

Classification Report:

               precision    recall  f1-score   support

   Benign (0)       0.96      0.99      0.97        72
Malignant (1)       0.97      0.93      0.95        42

     accuracy                           0.96       114
    macro avg       0.97      0.96      0.96       114
 weighted avg       0.97      0.96      0.96       114

2. RANDOM FOREST CLASSIFIER EVALUATION
Confusion Matrix:
  True Negatives (Benign correctly identified):    72
  False Positives (Benign mistaken for Malignant): 0
  False Negatives (Malignant missed):              3
  True Positives (Malignant correctly identified): 39

Classification Report:

               precision    recall  f1-score   support

   Benign (0)       0.96

## 7. Results Explanation & What Was Learned

### Analysis of Results
1. **Random Forest Classifier**:
   - **Test Accuracy**: **97.37%**
   - **Precision**: **100.00%** (0 false positives; no benign tumor was misdiagnosed as malignant)
   - **Recall**: **92.86%** (39 out of 42 malignant tumors correctly identified)
   - **F1-Score**: **96.30%**
2. **Logistic Regression**:
   - **Test Accuracy**: **96.49%**
   - **Precision**: **97.50%** (1 false positive)
   - **Recall**: **92.86%** (39 out of 42 malignant tumors correctly identified)
   - **F1-Score**: **95.12%**

Both models demonstrate strong predictive capability on this diagnostic dataset, with Random Forest achieving slightly higher overall accuracy and zero false positives.

### Key Takeaway / What Was Learned
In medical diagnostics, **Recall (Sensitivity)** is frequently the most critical metric because a False Negative (failing to detect a malignant cancer) carries potentially fatal consequences for the patient, whereas a False Positive can be caught with follow-up biopsies. Additionally, feature scaling is essential for linear models like Logistic Regression to prevent features with large numeric scales (e.g., `area_mean` ~1000s) from overpowering smaller features (e.g., `smoothness_mean` ~0.1), while tree-based models like Random Forest remain naturally invariant to monotonic feature scale differences.
